# ⚔️ FlyOpt vs PSO: Büyük Benchmark Çarpışması

Bu notebook, Biyolojik Sinek Motoru (FlyOpt T=20) ile piyasanın standart optimizasyon algoritmasını (PSO) **Sphere** ve **Rastrigin** görevlerinde GPU üzerinde yarıştırır.

In [ ]:
# 1. GEREKSİNİMLER VE DRIVE BAĞLANTISI
from google.colab import drive
import sys
import os
import torch
import numpy as np
import pandas as pd
import time
from scipy import sparse
import matplotlib.pyplot as plt
import seaborn as sns

drive.mount('/content/drive')

# Kendi Colab yapınıza uygun dizin ayarı (Lütfen gerekirse düzeltin)
project_path = '/content/drive/MyDrive/fly_op'
if not os.path.exists(project_path):
    project_path = '/content/drive/MyDrive/fly_op/fly_op'

sys.path.append(project_path)
sys.path.append(os.path.join(project_path, 'src'))
os.chdir(project_path)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Aktif Donanım: {device}')

In [ ]:
# 2. ALGORİTMALARIN TANIMLANMASI
from flyopt.variants.rate_brain import RateBrain, RateBrainConfig, build_subgraph_bfs, select_connected_encode_decode

# Benchmark Fonksiyonları
def sphere(x): return torch.sum(x**2, dim=1)
def rastrigin(x): return 10 * x.shape[1] + torch.sum(x**2 - 10 * torch.cos(2 * np.pi * x), dim=1)

# PSO Algoritması
def pso_optimize(func, dim=2, num_particles=1000, iters=150):
    x = (torch.rand((num_particles, dim), device=device) * 10 - 5)
    v = torch.zeros_like(x)
    pbest = x.clone()
    pbest_obj = func(x)
    gbest = pbest[torch.argmin(pbest_obj)].clone()
    gbest_obj = torch.min(pbest_obj)
    
    w, c1, c2 = 0.5, 1.5, 1.5
    history = []
    
    start = time.time()
    for _ in range(iters):
        r1 = torch.rand((num_particles, dim), device=device)
        r2 = torch.rand((num_particles, dim), device=device)
        v = w * v + c1 * r1 * (pbest - x) + c2 * r2 * (gbest - x)
        x = x + v
        obj = func(x)
        
        mask = obj < pbest_obj
        pbest[mask] = x[mask]
        pbest_obj[mask] = obj[mask]
        
        if torch.min(pbest_obj) < gbest_obj:
            gbest_obj = torch.min(pbest_obj)
        history.append(gbest_obj.item())
        
    return history, time.time() - start

# Biyolojik Motor (FlyOpt T=20)
def fly_optimize(func, brain, dim=2, num_particles=1000, iters=150):
    x = (torch.rand((num_particles, dim), device=device) * 10 - 5)
    best_obj = func(x)
    history = []
    input_pad = torch.zeros((num_particles, 4 - dim), device=device)
    
    start = time.time()
    with torch.no_grad():
        for _ in range(iters):
            env_input = torch.cat([x, input_pad], dim=1)
            fly_step = brain(env_input)
            move_vec = fly_step[:, :dim] * 0.1
            
            x_new = x - move_vec
            new_obj = func(x_new)
            
            mask = new_obj < best_obj
            x[mask] = x_new[mask]
            best_obj[mask] = new_obj[mask]
            
            history.append(torch.min(best_obj).item())
            
    return history, time.time() - start

In [ ]:
# 3. ERKEK SİNEK (Male CNS) YÜKLEMESİ
DATA_PROCESSED = f"{project_path}/data/processed"
print("Ağ yükleniyor...")
base_weights = sparse.load_npz(f"{DATA_PROCESSED}/malecns_adjacency.npz")
afferent = np.load(f"{DATA_PROCESSED}/malecns_afferent_indices.npy")
efferent = np.load(f"{DATA_PROCESSED}/malecns_efferent_indices.npy")

encode_full, decode_full = select_connected_encode_decode(base_weights, afferent, efferent, n_encode=4, n_decode=4, max_hops=6, seed=9000)
sub_real, encode_idx, decode_idx, _ = build_subgraph_bfs(base_weights, encode_full, decode_full, 3000, seed=9000)

cfg = RateBrainConfig(dim=2, n_readout=len(decode_idx), T=20, decode_scale=0.5, train_gain=True)
brain = RateBrain(sub_real, encode_idx, decode_idx, cfg, seed=42).to(device)
print("Biyolojik motor (T=20) GPU'ya yüklendi!")

In [ ]:
# 4. KAFES DÖVÜŞÜNÜ BAŞLAT VE GRAFİĞE DÖK
NUM_AGENTS = 5000
ITERS = 100

print(f"1. RAUNT: SPHERE (Kolay Görev - {NUM_AGENTS} Ajan)")
pso_s, time_pso_s = pso_optimize(sphere, num_particles=NUM_AGENTS, iters=ITERS)
fly_s, time_fly_s = fly_optimize(sphere, brain, num_particles=NUM_AGENTS, iters=ITERS)

print(f"\n2. RAUNT: RASTRIGIN (Zorlu/Tuzaklı Görev - {NUM_AGENTS} Ajan)")
pso_r, time_pso_r = pso_optimize(rastrigin, num_particles=NUM_AGENTS, iters=ITERS)
fly_r, time_fly_r = fly_optimize(rastrigin, brain, num_particles=NUM_AGENTS, iters=ITERS)

# ÇİZİM ALANI
sns.set_theme(style="darkgrid")
plt.figure(figsize=(14, 6))

# Sphere Grafiği
plt.subplot(1, 2, 1)
plt.plot(pso_s, label=f'PSO ({time_pso_s:.1f}s)', color='red', linewidth=2)
plt.plot(fly_s, label=f'FlyOpt T=20 ({time_fly_s:.1f}s)', color='blue', linewidth=2)
plt.title('Sphere (Ova) Optimizasyonu')
plt.xlabel('İterasyon')
plt.ylabel('Hata Payı (Düşük = İyi)')
plt.yscale('log')
plt.legend()

# Rastrigin Grafiği
plt.subplot(1, 2, 2)
plt.plot(pso_r, label=f'PSO ({time_pso_r:.1f}s)', color='red', linewidth=2)
plt.plot(fly_r, label=f'FlyOpt T=20 ({time_fly_r:.1f}s)', color='blue', linewidth=2)
plt.title('Rastrigin (Tuzaklı Dağ) Optimizasyonu')
plt.xlabel('İterasyon')
plt.ylabel('Hata Payı (Düşük = İyi)')
plt.yscale('log')
plt.legend()

plt.tight_layout()
plt.show()